In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Carpeta del Día 7.
CARPETA_DATOS = Path("datos_dia_07")
CARPETA_DATOS.mkdir(exist_ok=True)

# Base de datos del Día 7.
RUTA_DB = CARPETA_DATOS / "ventas_no_normalizadas.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_07

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_07\ventas_no_normalizadas.db


In [ ]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas_no_normalizadas (
    id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    fecha TEXT NOT NULL,

    cliente_nombre TEXT NOT NULL,
    cliente_correo TEXT NOT NULL,
    cliente_telefono TEXT,
    cliente_ciudad TEXT,

    producto_sku TEXT NOT NULL UNIQUE,
    producto_nombre TEXT NOT NULL,
    producto_categoria TEXT NOT NULL,
    precio_unitario REAL NOT NULL,
    cantidad INTEGER NOT NULL,
    subtotal REAL NOT NULL
)
""")

conexion.commit()
conexion.close()

print("Tabla ventas_no_normalizadas creada correctamente.")

Tabla ventas_no_normalizadas creada correctamente.


In [3]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

ventas = [
    (
        1, "2026-06-15",
        "Ana López", "ana@example.com", "555-111-2222", "CDMX",
        "P001", "Mouse inalámbrico", "Accesorios", 249.90, 2, 499.80
    ),
    (
        1, "2026-06-15",
        "Ana López", "ana@example.com", "555-111-2222", "CDMX",
        "P002", "Teclado mecánico", "Accesorios", 899.00, 1, 899.00
    ),
    (
        2, "2026-06-16",
        "Carlos Pérez", "carlos@example.com", "555-333-4444", "Guadalajara",
        "P003", "Monitor 24 pulgadas", "Pantallas", 3299.00, 1, 3299.00
    ),
    (
        3, "2026-06-16",
        "Ana López", "ana@example.com", "555-111-2222", "CDMX",
        "P004", "Cable HDMI", "Cables", 120.00, 3, 360.00
    ),
    (
        4, "2026-06-17",
        "María Torres", "maria@example.com", None, "Monterrey",
        "P001", "Mouse inalámbrico", "Accesorios", 249.90, 1, 249.90
    ),
    (
        5, "2026-06-17",
        "Ana Lopez", "ana.lopez@example.com", "555-111-2222", "CDMX",
        "P001", "Mouse inalámbrico", "Accesorios", 249.90, 1, 249.90
    )
]

for venta in ventas:
    cursor.execute("""
    INSERT INTO ventas_no_normalizadas (
        id_venta,
        fecha,
        cliente_nombre,
        cliente_correo,
        cliente_telefono,
        cliente_ciudad,
        producto_sku,
        producto_nombre,
        producto_categoria,
        precio_unitario,
        cantidad,
        subtotal
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, venta)

conexion.commit()
conexion.close()

print("Ventas insertadas correctamente.")

Ventas insertadas correctamente.


In [4]:
conexion = sqlite3.connect(RUTA_DB)

tabla_ventas = pd.read_sql_query("""
SELECT *
FROM ventas_no_normalizadas
ORDER BY id_registro
""", conexion)

conexion.close()

tabla_ventas

,id_registro,id_venta,fecha,cliente_nombre,cliente_correo,cliente_telefono,cliente_ciudad,producto_sku,producto_nombre,producto_categoria,precio_unitario,cantidad,subtotal
0,1,1,2026-06-15,Ana López,ana@example.com,555-111-2222,CDMX,P001,Mouse inalámbrico,Accesorios,249.9,2,499.8
1,2,1,2026-06-15,Ana López,ana@example.com,555-111-2222,CDMX,P002,Teclado mecánico,Accesorios,899.0,1,899.0
2,3,2,2026-06-16,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara,P003,Monitor 24 pulgadas,Pantallas,3299.0,1,3299.0
3,4,3,2026-06-16,Ana López,ana@example.com,555-111-2222,CDMX,P004,Cable HDMI,Cables,120.0,3,360.0
4,5,4,2026-06-17,María Torres,maria@example.com,NaN,Monterrey,P001,Mouse inalámbrico,Accesorios,249.9,1,249.9
5,6,5,2026-06-17,Ana Lopez,ana.lopez@example.com,555-111-2222,CDMX,P001,Mouse inalámbrico,Accesorios,249.9,1,249.9


In [5]:
conexion = sqlite3.connect(RUTA_DB)

repeticion_clientes = pd.read_sql_query("""
SELECT
    cliente_nombre,
    cliente_correo,
    COUNT(*) AS veces_repetido
FROM ventas_no_normalizadas
GROUP BY cliente_nombre, cliente_correo
ORDER BY veces_repetido DESC
""", conexion)

conexion.close()

repeticion_clientes

,cliente_nombre,cliente_correo,veces_repetido
0,Ana López,ana@example.com,3
1,Ana Lopez,ana.lopez@example.com,1
2,Carlos Pérez,carlos@example.com,1
3,María Torres,maria@example.com,1


In [6]:
conexion = sqlite3.connect(RUTA_DB)

ventas_ana = pd.read_sql_query("""
SELECT
    id_registro,
    id_venta,
    fecha,
    cliente_nombre,
    cliente_correo,
    producto_nombre,
    subtotal
FROM ventas_no_normalizadas
WHERE cliente_nombre LIKE 'Ana%'
ORDER BY id_registro
""", conexion)

conexion.close()

ventas_ana

,id_registro,id_venta,fecha,cliente_nombre,cliente_correo,producto_nombre,subtotal
0,1,1,2026-06-15,Ana López,ana@example.com,Mouse inalámbrico,499.8
1,2,1,2026-06-15,Ana López,ana@example.com,Teclado mecánico,899.0
2,4,3,2026-06-16,Ana López,ana@example.com,Cable HDMI,360.0
3,6,5,2026-06-17,Ana Lopez,ana.lopez@example.com,Mouse inalámbrico,249.9


In [7]:
conexion = sqlite3.connect(RUTA_DB)

posibles_inconsistencias_cliente = pd.read_sql_query("""
SELECT
    cliente_nombre,
    cliente_correo,
    cliente_telefono,
    cliente_ciudad,
    COUNT(*) AS apariciones
FROM ventas_no_normalizadas
WHERE cliente_nombre LIKE 'Ana%'
GROUP BY cliente_nombre, cliente_correo, cliente_telefono, cliente_ciudad
""", conexion)

conexion.close()

posibles_inconsistencias_cliente

,cliente_nombre,cliente_correo,cliente_telefono,cliente_ciudad,apariciones
0,Ana Lopez,ana.lopez@example.com,555-111-2222,CDMX,1
1,Ana López,ana@example.com,555-111-2222,CDMX,3


In [8]:
conexion = sqlite3.connect(RUTA_DB)

repeticion_productos = pd.read_sql_query("""
SELECT
    producto_sku,
    producto_nombre,
    producto_categoria,
    precio_unitario,
    COUNT(*) AS veces_vendido
FROM ventas_no_normalizadas
GROUP BY producto_sku, producto_nombre, producto_categoria, precio_unitario
ORDER BY veces_vendido DESC
""", conexion)

conexion.close()

repeticion_productos

,producto_sku,producto_nombre,producto_categoria,precio_unitario,veces_vendido
0,P001,Mouse inalámbrico,Accesorios,249.9,3
1,P002,Teclado mecánico,Accesorios,899.0,1
2,P003,Monitor 24 pulgadas,Pantallas,3299.0,1
3,P004,Cable HDMI,Cables,120.0,1


In [9]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

# Simulamos un error: solo actualizamos una fila de Ana.
cursor.execute("""
UPDATE ventas_no_normalizadas
SET cliente_correo = ?
WHERE id_registro = ?
""", ("ana.nuevo@example.com", 1))

conexion.commit()
conexion.close()

print("Se actualizó solo una fila de Ana.")

Se actualizó solo una fila de Ana.


In [10]:
conexion = sqlite3.connect(RUTA_DB)

correos_ana = pd.read_sql_query("""
SELECT
    id_registro,
    cliente_nombre,
    cliente_correo,
    producto_nombre,
    subtotal
FROM ventas_no_normalizadas
WHERE cliente_nombre LIKE 'Ana%'
ORDER BY id_registro
""", conexion)

conexion.close()

correos_ana

,id_registro,cliente_nombre,cliente_correo,producto_nombre,subtotal
0,1,Ana López,ana.nuevo@example.com,Mouse inalámbrico,499.8
1,2,Ana López,ana@example.com,Teclado mecánico,899.0
2,4,Ana López,ana@example.com,Cable HDMI,360.0
3,6,Ana Lopez,ana.lopez@example.com,Mouse inalámbrico,249.9


In [11]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

try:
    cursor.execute("""
    INSERT INTO ventas_no_normalizadas (
        id_venta,
        fecha,
        cliente_nombre,
        cliente_correo,
        cliente_telefono,
        cliente_ciudad,
        producto_sku,
        producto_nombre,
        producto_categoria,
        precio_unitario,
        cantidad,
        subtotal
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        None,
        None,
        "Luis Romero",
        "luis@example.com",
        "555-888-9999",
        "Puebla",
        None,
        None,
        None,
        None,
        None,
        None
    ))

    conexion.commit()
    print("Cliente insertado.")

except sqlite3.IntegrityError as error:
    print("No se pudo insertar el cliente sin venta.")
    print("Motivo:", error)

finally:
    conexion.close()

No se pudo insertar el cliente sin venta.
Motivo: NOT NULL constraint failed: ventas_no_normalizadas.id_venta


In [12]:
conexion = sqlite3.connect(RUTA_DB)

maria_antes = pd.read_sql_query("""
SELECT
    id_registro,
    id_venta,
    cliente_nombre,
    cliente_correo,
    cliente_telefono,
    cliente_ciudad,
    producto_nombre,
    subtotal
FROM ventas_no_normalizadas
WHERE cliente_nombre = 'María Torres'
""", conexion)

conexion.close()

maria_antes

,id_registro,id_venta,cliente_nombre,cliente_correo,cliente_telefono,cliente_ciudad,producto_nombre,subtotal
0,5,4,María Torres,maria@example.com,None,Monterrey,Mouse inalámbrico,249.9


In [ ]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()
cursor.execute("""
DELETE FROM ventas_no_normalizadas
WHERE cliente_nombre = ?
""", ("María Torres",))

conexion.commit()
conexion.close()


,id_registro,id_venta,cliente_nombre,cliente_correo,cliente_telefono,cliente_ciudad,producto_nombre,subtotal
0,5,4,María Torres,maria@example.com,None,Monterrey,Mouse inalámbrico,249.9


In [ ]:
conexion = sqlite3.connect(RUTA_DB)

maria_despues = pd.read_sql_query("""
SELECT
    id_registro,
    cliente_nombre,
    cliente_correo,
    cliente_telefono,
    cliente_ciudad
FROM ventas_no_normalizadas
WHERE cliente_nombre = 'María Torres'
""", conexion)

conexion.close()

maria_despues

,id_registro,cliente_nombre,cliente_correo,cliente_telefono,cliente_ciudad
0,5,María Torres,maria@example.com,None,Monterrey


In [15]:
clientes = pd.DataFrame([
    {"id_cliente": 1, "nombre": "Ana López", "correo": "ana@example.com", "telefono": "555-111-2222", "ciudad": "CDMX"},
    {"id_cliente": 2, "nombre": "Carlos Pérez", "correo": "carlos@example.com", "telefono": "555-333-4444", "ciudad": "Guadalajara"},
    {"id_cliente": 3, "nombre": "María Torres", "correo": "maria@example.com", "telefono": None, "ciudad": "Monterrey"},
])

productos = pd.DataFrame([
    {"id_producto": 1, "sku": "P001", "nombre": "Mouse inalámbrico", "categoria": "Accesorios", "precio": 249.90},
    {"id_producto": 2, "sku": "P002", "nombre": "Teclado mecánico", "categoria": "Accesorios", "precio": 899.00},
    {"id_producto": 3, "sku": "P003", "nombre": "Monitor 24 pulgadas", "categoria": "Pantallas", "precio": 3299.00},
    {"id_producto": 4, "sku": "P004", "nombre": "Cable HDMI", "categoria": "Cables", "precio": 120.00},
])

ventas = pd.DataFrame([
    {"id_venta": 1, "fecha": "2026-06-15", "id_cliente": 1},
    {"id_venta": 2, "fecha": "2026-06-16", "id_cliente": 2},
    {"id_venta": 3, "fecha": "2026-06-16", "id_cliente": 1},
    {"id_venta": 4, "fecha": "2026-06-17", "id_cliente": 3},
])

detalle_ventas = pd.DataFrame([
    {"id_detalle": 1, "id_venta": 1, "id_producto": 1, "cantidad": 2, "precio_unitario": 249.90},
    {"id_detalle": 2, "id_venta": 1, "id_producto": 2, "cantidad": 1, "precio_unitario": 899.00},
    {"id_detalle": 3, "id_venta": 2, "id_producto": 3, "cantidad": 1, "precio_unitario": 3299.00},
    {"id_detalle": 4, "id_venta": 3, "id_producto": 4, "cantidad": 3, "precio_unitario": 120.00},
    {"id_detalle": 5, "id_venta": 4, "id_producto": 1, "cantidad": 1, "precio_unitario": 249.90},
])

print("Clientes")
display(clientes)

print("Productos")
display(productos)

print("Ventas")
display(ventas)

print("Detalle de ventas")
display(detalle_ventas)

Clientes


,id_cliente,nombre,correo,telefono,ciudad
0,1,Ana López,ana@example.com,555-111-2222,CDMX
1,2,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara
2,3,María Torres,maria@example.com,NaN,Monterrey


Productos


,id_producto,sku,nombre,categoria,precio
0,1,P001,Mouse inalámbrico,Accesorios,249.9
1,2,P002,Teclado mecánico,Accesorios,899.0
2,3,P003,Monitor 24 pulgadas,Pantallas,3299.0
3,4,P004,Cable HDMI,Cables,120.0


Ventas


,id_venta,fecha,id_cliente
0,1,2026-06-15,1
1,2,2026-06-16,2
2,3,2026-06-16,1
3,4,2026-06-17,3


Detalle de ventas


,id_detalle,id_venta,id_producto,cantidad,precio_unitario
0,1,1,1,2,249.9
1,2,1,2,1,899.0
2,3,2,3,1,3299.0
3,4,3,4,3,120.0
4,5,4,1,1,249.9


In [16]:
ventas_con_cliente = ventas.merge(
    clientes,
    on="id_cliente",
    how="left"
)

detalle_con_producto = detalle_ventas.merge(
    productos,
    on="id_producto",
    how="left"
)

detalle_con_producto["subtotal"] = (
    detalle_con_producto["cantidad"] * detalle_con_producto["precio_unitario"]
)

venta_completa = detalle_con_producto.merge(
    ventas_con_cliente,
    on="id_venta",
    how="left"
)

venta_completa = venta_completa[
    [
        "id_venta",
        "fecha",
        "nombre_y",
        "correo",
        "sku",
        "nombre_x",
        "categoria",
        "cantidad",
        "precio_unitario",
        "subtotal"
    ]
]

venta_completa = venta_completa.rename(columns={
    "nombre_y": "cliente",
    "nombre_x": "producto"
})

venta_completa

,id_venta,fecha,cliente,correo,sku,producto,categoria,cantidad,precio_unitario,subtotal
0,1,2026-06-15,Ana López,ana@example.com,P001,Mouse inalámbrico,Accesorios,2,249.9,499.8
1,1,2026-06-15,Ana López,ana@example.com,P002,Teclado mecánico,Accesorios,1,899.0,899.0
2,2,2026-06-16,Carlos Pérez,carlos@example.com,P003,Monitor 24 pulgadas,Pantallas,1,3299.0,3299.0
3,3,2026-06-16,Ana López,ana@example.com,P004,Cable HDMI,Cables,3,120.0,360.0
4,4,2026-06-17,María Torres,maria@example.com,P001,Mouse inalámbrico,Accesorios,1,249.9,249.9


In [17]:
conexion = sqlite3.connect(RUTA_DB)

total_filas_no_normalizada = pd.read_sql_query("""
SELECT COUNT(*) AS filas_tabla_unica
FROM ventas_no_normalizadas
""", conexion)

clientes_repetidos = pd.read_sql_query("""
SELECT
    cliente_nombre,
    COUNT(*) AS apariciones
FROM ventas_no_normalizadas
GROUP BY cliente_nombre
ORDER BY apariciones DESC
""", conexion)

productos_repetidos = pd.read_sql_query("""
SELECT
    producto_sku,
    producto_nombre,
    COUNT(*) AS apariciones
FROM ventas_no_normalizadas
GROUP BY producto_sku, producto_nombre
ORDER BY apariciones DESC
""", conexion)

conexion.close()

print("Total de filas en tabla no normalizada")
display(total_filas_no_normalizada)

print("Repetición de clientes")
display(clientes_repetidos)

print("Repetición de productos")
display(productos_repetidos)

Total de filas en tabla no normalizada


,filas_tabla_unica
0,6


Repetición de clientes


,cliente_nombre,apariciones
0,Ana López,3
1,María Torres,1
2,Carlos Pérez,1
3,Ana Lopez,1


Repetición de productos


,producto_sku,producto_nombre,apariciones
0,P001,Mouse inalámbrico,3
1,P002,Teclado mecánico,1
2,P003,Monitor 24 pulgadas,1
3,P004,Cable HDMI,1


In [6]:
from pathlib import Path
import sqlite3
import pandas as pd

CARPETA_DATOS = Path("datos_dia_07")
CARPETA_DATOS.mkdir(exist_ok=True)

RUTA_DB_03 = CARPETA_DATOS / "tabla_reto_D3.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB_03.resolve())

conexion = sqlite3.connect(RUTA_DB_03)
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tabla_reto_D3 (
    id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    fecha TEXT NOT NULL,

    cliente_nombre TEXT NOT NULL,
    cliente_correo TEXT NOT NULL,
    cliente_telefono TEXT,
    cliente_ciudad TEXT,

    producto_sku TEXT NOT NULL,
    producto_nombre TEXT NOT NULL,
    producto_categoria TEXT NOT NULL,
    precio_unitario REAL NOT NULL,
    cantidad INTEGER NOT NULL,
    subtotal REAL NOT NULL
)
""")

conexion.commit()
conexion.close()

print("Tabla tabla_reto_3 creada correctamente.")

conexion = sqlite3.connect(RUTA_DB_03)
cursor = conexion.cursor()

ventas = [
    (
        1, "2026-06-15",
        "Armando Guerra", "camaraman@example.com", "555-111-6969", "CDMX",
        "P001", "Mouse inalámbrico", "Accesorios", 249.90, 2, 499.80
    ),
    (
        1, "2026-06-15",
        "Mayo Cara", "mayo@example.com", "555-111-3269", "CDMX",
        "P002", "Teclado mecánico", "Accesorios", 899.00, 1, 899.00
    ),
    (
        2, "2026-06-16",
        "Tristan Rangel", "tristan@example.com", "555-325-4444", "Guadalajara",
        "P003", "Xbox Series S", "Pantallas", 3299.00, 1, 3299.00
    ),
    (
        3, "2026-06-16",
        "Chuy Malboro", "chuy@example.com", "555-111-4589", "CDMX",
        "P004", "Cable HDMI", "Cables", 120.00, 3, 360.00
    ),
    (
        4, "2026-06-17",
        "Filiberto Cabrera", "fili@example.com", None, "Monterrey",
        "P001", "Mouse inalámbrico", "Accesorios", 249.90, 1, 249.90
    ),
    (
        5, "2026-06-17",
        "Tristan Rangel", "tristan@example.com", "555-325-4444", "Guadalajara",
        "P001", "Mouse inalámbrico", "Accesorios", 249.90, 1, 249.90
    )
]

for venta in ventas:
    cursor.execute("""
    INSERT INTO tabla_reto_D3 (
        id_venta,
        fecha,
        cliente_nombre,
        cliente_correo,
        cliente_telefono,
        cliente_ciudad,
        producto_sku,
        producto_nombre,
        producto_categoria,
        precio_unitario,
        cantidad,
        subtotal
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, venta)

conexion.commit()
conexion.close()

print("Ventas insertadas correctamente.")

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_07

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_07\tabla_reto_D3.db
Tabla tabla_reto_3 creada correctamente.
Ventas insertadas correctamente.
